In [ ]:
from google.colab import drive
import pickle
import pandas as pd
from collections import defaultdict
import random
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
# Copying 8k to Colabs local SSD
if not os.path.exists("/content/flickr8k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr8k.zip" /content/

In [ ]:
if not os.path.isdir("/content/flickr8k/Images"):
  !unzip "/content/flickr8k.zip" -d "/content/flickr8k"

In [ ]:
print("Images:", len(os.listdir("/content/flickr8k/Images")))

Images: 8091


In [ ]:
# Copying 30k to Colabs local SSD
if not os.path.exists("/content/flickr30k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr30k.zip" /content/

In [ ]:
if not os.path.isdir("/content/flickr30k/Images"):
 !unzip "/content/flickr30k.zip" -d "/content/flickr30k"

In [ ]:
print("Images:", len(os.listdir("/content/flickr30k/Images")))

Images: 31811


In [ ]:
DATASETS = {
    "flickr8k": {
        "ROOT": "/content/flickr8k",
        "IMAGE_DIR": "/content/flickr8k/Images",
        "CAPTION_FILE": "/content/flickr8k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/flickr8k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/vocab.pkl"
    },
    "flickr30k": {
        "ROOT": "/content/flickr30k",
        "IMAGE_DIR": "/content/flickr30k/Images",
        "CAPTION_FILE": "/content/flickr30k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/flickr30k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/vocab.pkl"
    }
}

In [ ]:
def image_caption_map(train_df,val_df,test_df):
  train_caption_map = defaultdict(list)
  val_caption_map = defaultdict(list)
  test_caption_map = defaultdict(list)

  bad_img = "861608773_bdafd5c996.jpg"

  train_caption_map.pop(bad_img, None)
  val_caption_map.pop(bad_img, None)
  test_caption_map.pop(bad_img, None)

  for _, row in train_df.iterrows():
      train_caption_map[row["image"]].append(row["caption"])

  for _, row in val_df.iterrows():
      val_caption_map[row["image"]].append(row["caption"])

  for _, row in test_df.iterrows():
      test_caption_map[row["image"]].append(row["caption"])
  return train_caption_map,val_caption_map,test_caption_map



In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Encoding

def encode_caption(text, vocab):

    tokens = text.split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [ ]:
class FlickrRetrievalDataset(Dataset):
    def __init__(self, caption_map, image_dir, vocab, transform=None, random_caption=True):
        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.random_caption = random_caption

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]

        captions = self.caption_map[image_name]

        if self.random_caption:
            caption = random.choice(captions)
        else:
            caption = captions[0]      # fixed caption for eval

        try:
          image = Image.open(
              os.path.join(self.image_dir, image_name)
          ).convert("RGB")

        except Exception:
            return self.__getitem__(
                (idx + 1) % len(self)
            )

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [ ]:
class FlickrAllCaptionEvalDataset(Dataset):

    def __init__(self, caption_map, image_dir, vocab, transform=None):

        self.samples = []
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform

        for image_name in sorted(caption_map.keys()):

            for caption in caption_map[image_name]:

                self.samples.append(
                    (image_name, caption)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_name, caption = self.samples[idx]

        image = Image.open(
            os.path.join(self.image_dir, image_name)
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []
    captions = []
    lengths = []

    for image, caption in batch:
        images.append(image)
        captions.append(caption)
        lengths.append(len(caption))

    images = torch.stack(images)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    lengths = torch.tensor(lengths)

    return images, captions, lengths

In [ ]:
def prepare_datasets_dataloaders(train_caption_map,val_caption_map,test_caption_map):

  train_dataset = FlickrRetrievalDataset(train_caption_map,IMAGE_DIR,vocab,train_transform,random_caption=True)
  val_dataset = FlickrRetrievalDataset(val_caption_map,IMAGE_DIR,vocab,image_transform,random_caption=False)
  test_dataset = FlickrRetrievalDataset(test_caption_map,IMAGE_DIR,vocab,image_transform,random_caption=False)
  all_caption_test_dataset = FlickrAllCaptionEvalDataset(test_caption_map,IMAGE_DIR,vocab,image_transform)

  BATCH_SIZE = 256
  train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=8,
    pin_memory=True,
    )
  val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=8,
        pin_memory=True,
    )
  test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=8,
        pin_memory=True,
    )
  all_caption_test_loader = DataLoader(
    all_caption_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=8
    )
  return train_loader,val_loader,test_loader,all_caption_test_loader

ViT(vit_base_patch16_224) Image Encoder

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F


class ViTEncoder(nn.Module):

    def __init__(
        self,
        embed_dim=512,
        freeze_backbone=True,
        dropout=0.15
    ):
        super().__init__()

        self.freeze_backbone = freeze_backbone

        # --------------------------------------------------
        # ViT-B/16
        # Original ImageNet pretrained ViT-B/16
        # --------------------------------------------------

        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )

        # --------------------------------------------------
        # Freeze / unfreeze backbone
        # --------------------------------------------------

        for param in self.vit.parameters():
            param.requires_grad = not freeze_backbone

        # --------------------------------------------------
        # Projection Head
        # 768 -> 1024 -> 512
        # --------------------------------------------------

        self.projection = nn.Sequential(

            nn.Linear(768, 1024),

            nn.LayerNorm(1024),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(1024, embed_dim),

            nn.LayerNorm(embed_dim)
        )

        # --------------------------------------------------
        # Initialization
        # --------------------------------------------------

        for module in self.projection:

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def forward(self, images):

        # --------------------------------------------------
        # ViT feature extraction
        # --------------------------------------------------

        if self.freeze_backbone:

            with torch.no_grad():

                features = self.vit.forward_features(
                    images
                )

        else:

            features = self.vit.forward_features(
                images
            )

        # --------------------------------------------------
        # ViT output
        #
        # (B, 197, 768)
        #
        # 1 CLS token
        # 196 patch tokens
        # --------------------------------------------------

        cls_token = features[:, 0]

        # --------------------------------------------------
        # Patch tokens
        # --------------------------------------------------

        patch_tokens = features[:, 1:]

        # --------------------------------------------------
        # Mean pooled patch representation
        # --------------------------------------------------

        patch_mean = patch_tokens.mean(
            dim=1
        )

        # --------------------------------------------------
        # Combine CLS + patch information
        # --------------------------------------------------

        features = (
            cls_token +
            patch_mean
        ) / 2.0

        # --------------------------------------------------
        # Projection
        # --------------------------------------------------

        embeddings = self.projection(
            features
        )

        # --------------------------------------------------
        # L2 normalization
        # --------------------------------------------------

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

**1-layer BiLSTM,the caption is processed in both directions to yields better sentence representations**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)


class TextEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=300,
        hidden_dim=512,
        attention_dim=512,
        output_dim=512,
        pad_idx=0,
        dropout=0.2
    ):

        super().__init__()

        self.pad_idx = pad_idx

        # =========================================================
        # Word Embedding
        # =========================================================

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        # =========================================================
        # Bidirectional LSTM
        # =========================================================

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
            dropout=0
        )

        # =========================================================
        # Attention
        # =========================================================

        self.attention_projection = nn.Linear(
            hidden_dim * 2,
            attention_dim
        )

        self.attention_score = nn.Linear(
            attention_dim,
            1
        )

        # =========================================================
        # Projection Head
        # =========================================================

        self.projection = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                1024
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                1024,
                output_dim
            )
        )

        # =========================================================
        # Initialization
        # =========================================================

        nn.init.xavier_uniform_(
            self.embedding.weight
        )

        with torch.no_grad():
            self.embedding.weight[pad_idx].zero_()

        for module in self.modules():

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

            elif isinstance(module, nn.LSTM):

                for name, param in module.named_parameters():

                    if "weight_ih" in name:
                        nn.init.xavier_uniform_(
                            param
                        )

                    elif "weight_hh" in name:
                        nn.init.orthogonal_(
                            param
                        )

                    elif "bias" in name:
                        nn.init.zeros_(
                            param
                        )

                        # Forget gate bias
                        # [i, f, g, o]
                        n = param.size(0)

                        param.data[
                            n // 4:n // 2
                        ].fill_(1.0)

    def forward(
        self,
        captions,
        lengths
    ):

        device = captions.device

        lengths = lengths.to(
            device,
            non_blocking=True
        )


        # =========================================================
        # Embedding
        # =========================================================

        embedded = self.embedding(
            captions
        )

        # embedded:
        # (B, T, 300)

        # =========================================================
        # Packed LSTM
        # =========================================================

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.lstm(
            packed
        )

        # =========================================================
        # Restore sequence
        # =========================================================

        lstm_output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True
        )

        # lstm_output:
        # (B, T, 1024)

        # =========================================================
        # Attention
        # =========================================================

        attention_features = torch.tanh(
            self.attention_projection(
                lstm_output
            )
        )

        attention_scores = self.attention_score(
            attention_features
        ).squeeze(-1)

        # =========================================================
        # Mask padding
        # =========================================================

        max_len = lstm_output.size(1)

        mask = (
            torch.arange(
                max_len,
                device=captions.device
            )[None, :]
            < lengths[:, None]
        )

        attention_scores = attention_scores.masked_fill(
            ~mask,
            float("-inf")
        )

        # =========================================================
        # Attention weights
        # =========================================================

        alpha = F.softmax(
            attention_scores,
            dim=1
        )

        # =========================================================
        # Attention pooling
        # =========================================================

        sentence_embedding = torch.sum(
            lstm_output *
            alpha.unsqueeze(-1),
            dim=1
        )

        # sentence_embedding:
        # (B, 1024)

        # =========================================================
        # Projection
        # =========================================================

        embeddings = self.projection(
            sentence_embedding
        )

        # =========================================================
        # L2 normalization
        # =========================================================

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [ ]:
# ============================================================
# Joint Model
# ViT + Attention BiLSTM Retrieval
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class ViTLSTMRetrieval(nn.Module):

    def __init__(
        self,
        image_encoder,
        text_encoder,
        initial_temperature=0.07
    ):

        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # ----------------------------------------------------
        # Learnable CLIP-style logit scale
        # ----------------------------------------------------

        self.logit_scale = nn.Parameter(
            torch.tensor(
                torch.log(
                    torch.tensor(
                        1.0 / initial_temperature
                    )
                )
            )
        )

    def forward(
        self,
        images,
        captions,
        lengths
    ):

        # ----------------------------------------------------
        # Image encoder
        # ----------------------------------------------------

        image_emb = self.image_encoder(
            images
        )

        # ----------------------------------------------------
        # Text encoder
        # ----------------------------------------------------

        text_emb = self.text_encoder(
            captions,
            lengths
        )

        # ----------------------------------------------------
        # Explicit L2 normalization
        # ----------------------------------------------------

        image_emb = F.normalize(
            image_emb,
            p=2,
            dim=1
        )

        text_emb = F.normalize(
            text_emb,
            p=2,
            dim=1
        )

        return image_emb, text_emb

    def get_logit_scale(self):

        # Prevent temperature from becoming
        # excessively large during training.

        return self.logit_scale.exp().clamp(
            max=100
        )

In [ ]:
def build_image_text_encoder(vocab, device):

    # ============================================================
    # IMAGE ENCODER
    # ============================================================

    image_encoder = ViTEncoder(
        freeze_backbone=True
    ).to(device)


    # ============================================================
    # TEXT ENCODER
    # ============================================================

    text_encoder = TextEncoder(
        vocab_size=len(vocab),
        embed_dim=300,
        hidden_dim=512,
        pad_idx=vocab["<PAD>"]
    ).to(device)


    # ============================================================
    # JOINT MODEL
    # ============================================================

    model = ViTLSTMRetrieval(
        image_encoder=image_encoder,
        text_encoder=text_encoder,
        initial_temperature=0.07
    ).to(device)


    return model

In [ ]:
def clip_contrastive_loss(
    image_emb,
    text_emb,
    logit_scale
):

    # ------------------------------------------------------------
    # Normalize again for safety
    # ------------------------------------------------------------

    image_emb = F.normalize(
        image_emb,
        p=2,
        dim=1
    )

    text_emb = F.normalize(
        text_emb,
        p=2,
        dim=1
    )

    # ------------------------------------------------------------
    # Learnable temperature
    # ------------------------------------------------------------

    logit_scale = logit_scale.exp().clamp(max=100)

    # ------------------------------------------------------------
    # Similarity matrix
    # ------------------------------------------------------------

    logits = (
        image_emb @ text_emb.T
    ) * logit_scale

    # ------------------------------------------------------------
    # Ground-truth matching indices
    # ------------------------------------------------------------

    targets = torch.arange(
        image_emb.size(0),
        device=image_emb.device
    )

    # ------------------------------------------------------------
    # Image → Text
    # ------------------------------------------------------------

    loss_i2t = F.cross_entropy(
        logits,
        targets
    )

    # ------------------------------------------------------------
    # Text → Image
    # ------------------------------------------------------------

    loss_t2i = F.cross_entropy(
        logits.T,
        targets
    )

    # ------------------------------------------------------------
    # Bidirectional InfoNCE
    # ------------------------------------------------------------

    loss = (
        loss_i2t +
        loss_t2i
    ) / 2

    return loss, logits

In [ ]:
# Training
def define_optimizer(model):
  optimizer = torch.optim.AdamW(
    [
        {
            "params": model.image_encoder.vit.blocks[-2:].parameters(),
            "lr": 1e-5
        },
        {
            "params": model.image_encoder.vit.norm.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.image_encoder.projection.parameters(),
            "lr": 2e-4
        },
        {
            "params": model.text_encoder.parameters(),
            "lr": 2e-4
        },
        {
            "params": [model.logit_scale],
            "lr": 5e-5
        }
    ],
    weight_decay=1e-4
)
  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=50)
  return optimizer,scheduler

In [ ]:
def evaluate(model, dataloader, device):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for images, captions, lengths in dataloader:

            images = images.to(
            device,
            non_blocking=True
            )

            captions = captions.to(
                device,
                non_blocking=True
            )

            lengths = lengths.to(
                device,
                non_blocking=True
            )

            image_emb, text_emb = model(
                images,
                captions,
                lengths
            )

            loss, _ = clip_contrastive_loss(
                image_emb,
                text_emb,
                model.logit_scale
            )

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / total_samples

In [ ]:
def model_training(model,train_loader,val_loader,optimizer,scheduler,dataset_name):
  import time
  import torch

  # Training
  NUM_EPOCHS = 50
  best_val_loss = float("inf")
  patience = 8
  epochs_without_improvement = 0

  SAVE_DIR = f"/content/drive/MyDrive/MMRetrieval/R4/{dataset_name}"
  os.makedirs(SAVE_DIR, exist_ok=True)

  BEST_MODEL_PATH = os.path.join(
      SAVE_DIR,
      "best_retrieval_R4_model.pth"
  )

  for epoch in range(NUM_EPOCHS):

      print(f"\n================ Epoch {epoch+1}/{NUM_EPOCHS} ================")

      model.train()
      running_loss = 0.0

      epoch_start = time.time()

      for batch_idx, (images, captions, lengths) in enumerate(train_loader):

          batch_start = time.time()

          images = images.to(device, non_blocking=True)
          captions = captions.to(device, non_blocking=True)
          lengths = lengths.to(device,non_blocking=True)
          optimizer.zero_grad(set_to_none=True)

          image_emb, text_emb = model(
              images,
              captions,
              lengths
          )

          loss, _ = clip_contrastive_loss(
              image_emb,
              text_emb,
              model.logit_scale
          )

          loss.backward()

          torch.nn.utils.clip_grad_norm_(
              model.parameters(),
              max_norm=1.0
          )

          optimizer.step()

          running_loss += loss.item()

          batch_time = time.time() - batch_start

          print(
              f"Batch {batch_idx+1:02d}/{len(train_loader)} | "
              f"Loss: {loss.item():.4f} | "
              f"Time: {batch_time:.2f}s"
          )

      train_time = time.time() - epoch_start

      train_loss = running_loss / len(train_loader)

      # ---------------- Validation ----------------
      val_start = time.time()

      val_loss = evaluate(
          model,
          val_loader,
          device
      )

      val_time = time.time() - val_start

      scheduler.step()

      print("\n---------------- Summary ----------------")
      print(f"Train Loss     : {train_loss:.4f}")
      print(f"Validation Loss: {val_loss:.4f}")
      print(f"Training Time  : {train_time:.2f} sec")
      print(f"Validation Time: {val_time:.2f} sec")
      print(
          f"GPU Memory Used: "
          f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
      )

      # Save best model
      if val_loss < best_val_loss:

          best_val_loss = val_loss
          epochs_without_improvement = 0

          torch.save(
              {
                  "model_state_dict": model.state_dict(),
                  "optimizer_state_dict": optimizer.state_dict(),
                  "epoch": epoch,
                  "scheduler_state_dict": scheduler.state_dict(),
                  "val_loss": val_loss
              },
              BEST_MODEL_PATH
          )
          print(f"✓ Best model saved(Val Loss: {val_loss:.4f})")
      else:
        epochs_without_improvement += 1
        print(f"No improvement for "f"{epochs_without_improvement}/{patience} epochs")

      if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered!")
        break

  FINAL_MODEL_PATH = BEST_MODEL_PATH
  checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
  model.load_state_dict(checkpoint["model_state_dict"])

  return model,FINAL_MODEL_PATH

In [ ]:
def extract_embeddings(model, dataloader, device):

    model.eval()

    image_embeddings = []
    text_embeddings = []

    with torch.no_grad():

        for images, captions, lengths in dataloader:

            images = images.to(
                device,
                non_blocking=True
            )

            captions = captions.to(
                device,
                non_blocking=True
            )
            lengths = lengths.to(
                device,
                non_blocking=True
            )

            img_emb, txt_emb = model(
                images,
                captions,
                lengths
            )

            image_embeddings.append(
                img_emb.cpu()
            )

            text_embeddings.append(
                txt_emb.cpu()
            )

    image_embeddings = torch.cat(
        image_embeddings,
        dim=0
    )

    text_embeddings = torch.cat(
        text_embeddings,
        dim=0
    )

    return (
        image_embeddings,
        text_embeddings
    )

In [ ]:
# image -> text
def image_to_text_recall(similarity, k):

    correct = 0

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        topk = similarity[img_idx].topk(k).indices.tolist()

        if any(idx in gt_caps for idx in topk):
            correct += 1

    return correct / similarity.shape[0]


In [ ]:
def text_to_image_recall(similarity, k):

    similarity_t = similarity.T

    correct = 0

    for cap_idx in range(similarity_t.shape[0]):

        gt_img = cap_idx // 5

        topk = similarity_t[cap_idx]\
            .topk(k)\
            .indices\
            .tolist()

        if gt_img in topk:
            correct += 1

    return correct / similarity_t.shape[0]


In [ ]:
def image_to_text_mrr(similarity):

    reciprocal_ranks = []

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        sorted_idx = torch.argsort(
            similarity[img_idx],
            descending=True
        )

        best_rank = float("inf")

        for cap in gt_caps:

            rank = (
                (sorted_idx == cap)
                .nonzero(as_tuple=True)[0]
                .item()
            ) + 1

            best_rank = min(best_rank, rank)

        reciprocal_ranks.append(
            1.0 / best_rank
        )

    return np.mean(reciprocal_ranks)

In [ ]:
def text_to_image_mrr(similarity):

    similarity_t2i = similarity.T

    reciprocal_ranks = []

    for cap_idx in range(similarity_t2i.shape[0]):

        gt_image = cap_idx // 5

        sorted_idx = torch.argsort(
            similarity_t2i[cap_idx],
            descending=True
        )

        rank = (
            (sorted_idx == gt_image)
            .nonzero(as_tuple=True)[0]
            .item()
        ) + 1

        reciprocal_ranks.append(
            1.0 / rank
        )

    return np.mean(reciprocal_ranks)

In [ ]:
import numpy as np
def model_testing(model,FINAL_MODEL_PATH,all_caption_test_loader):

  model.eval()
  with torch.inference_mode():
    image_embs, text_embs = extract_embeddings(
      model,
      all_caption_test_loader,
      device
    )
  print(image_embs.shape)
  print(text_embs.shape)
  # Similarity Matrix
  unique_image_embs = image_embs[::5]

  similarity = unique_image_embs @ text_embs.T

  print("Image embeddings:", unique_image_embs.shape)
  print("Text embeddings:", text_embs.shape)
  print("Similarity:", similarity.shape)
  results = pd.DataFrame({
      "Metric": [
          "Recall@1",
          "Recall@5",
          "Recall@10",
          "MRR"
      ],
      "Image→Text": [
          image_to_text_recall(similarity, 1),
          image_to_text_recall(similarity, 5),
          image_to_text_recall(similarity, 10),
          image_to_text_mrr(similarity)
      ],
      "Text→Image": [
          text_to_image_recall(similarity, 1),
          text_to_image_recall(similarity, 5),
          text_to_image_recall(similarity, 10),
          text_to_image_mrr(similarity)
      ]
  })

  results["Image→Text"] = results["Image→Text"].round(6)
  results["Text→Image"] = results["Text→Image"].round(6)

  display(results)
  return

In [ ]:
import gc
def reset_resourses():
  gc.collect()
  torch.cuda.empty_cache()
  print("\nMemory after cleanup")
  print("Allocated:",torch.cuda.memory_allocated()/1024**3)
  print("Reserved:",torch.cuda.memory_reserved()/1024**3)

In [ ]:
for dataset_name, cfg in DATASETS.items():
    torch.cuda.reset_peak_memory_stats()
    print("\n************************")
    print(f"\nProcessing - {dataset_name}")
    print("\n************************\n")
    ROOT = cfg["ROOT"]
    IMAGE_DIR = cfg["IMAGE_DIR"]
    CAPTION_FILE = cfg["CAPTION_FILE"]
    with open(cfg["flickr_split"],"rb") as f:
      split = pickle.load(f)

    with open(cfg["vocab"],"rb") as g:
      vocab = pickle.load(g)

    model = build_image_text_encoder(vocab, device)

   # print("MODEL\n",model)

    optimizer,scheduler = define_optimizer(model)

    df = pd.read_csv(CAPTION_FILE)
    train_imgs = split["train"]
    val_imgs = split["val"]
    test_imgs = split["test"]

    print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

    train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
    val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
    test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

    train_df = train_df.dropna(subset=["caption"]).reset_index(drop=True)
    val_df = val_df.dropna(subset=["caption"]).reset_index(drop=True)
    test_df = test_df.dropna(subset=["caption"]).reset_index(drop=True)

    train_caption_map,val_caption_map,test_caption_map = image_caption_map(train_df,val_df,test_df)

    train_loader,val_loader,test_loader,all_caption_test_loader = prepare_datasets_dataloaders(train_caption_map,val_caption_map,test_caption_map)

    best_model,FINAL_MODEL_PATH  = model_training(model,train_loader,val_loader,optimizer,scheduler,dataset_name)

    model_testing(best_model,FINAL_MODEL_PATH,all_caption_test_loader)

    del model
    del optimizer
    del scheduler

    del train_loader
    del val_loader
    del test_loader
    del all_caption_test_loader

    del train_df
    del val_df
    del test_df

    del train_caption_map
    del val_caption_map
    del test_caption_map

    del split
    del vocab
    del df

    reset_resourses()


************************

Processing - flickr8k

************************



/tmp/ipykernel_17484/1019136155.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(


Train: 6068 | Val: 1011 | Test: 1012

================ Epoch 1/50 ================
Batch 01/24 | Loss: 5.6760 | Time: 0.61s
Batch 02/24 | Loss: 5.6173 | Time: 0.54s
Batch 03/24 | Loss: 5.5494 | Time: 0.54s
Batch 04/24 | Loss: 5.5041 | Time: 0.54s
Batch 05/24 | Loss: 5.4214 | Time: 0.56s
Batch 06/24 | Loss: 5.4004 | Time: 0.54s
Batch 07/24 | Loss: 5.3053 | Time: 0.54s
Batch 08/24 | Loss: 5.3395 | Time: 0.54s
Batch 09/24 | Loss: 5.2720 | Time: 0.54s
Batch 10/24 | Loss: 5.0972 | Time: 0.54s
Batch 11/24 | Loss: 5.0318 | Time: 0.56s
Batch 12/24 | Loss: 5.0022 | Time: 0.54s
Batch 13/24 | Loss: 5.0215 | Time: 0.54s
Batch 14/24 | Loss: 4.8819 | Time: 0.54s
Batch 15/24 | Loss: 4.7988 | Time: 0.55s
Batch 16/24 | Loss: 4.8989 | Time: 0.54s
Batch 17/24 | Loss: 4.7910 | Time: 0.54s
Batch 18/24 | Loss: 4.7306 | Time: 0.53s
Batch 19/24 | Loss: 4.6904 | Time: 0.53s
Batch 20/24 | Loss: 4.6881 | Time: 0.53s
Batch 21/24 | Loss: 4.6511 | Time: 0.54s
Batch 22/24 | Loss: 4.5228 | Time: 0.53s
Batch 23/24 | L

,Metric,Image→Text,Text→Image
0,Recall@1,0.305336,0.240316
1,Recall@5,0.622530,0.553360
2,Recall@10,0.762846,0.686364
3,MRR,0.450416,0.384648



Memory after cleanup
Allocated: 0.7723188400268555
Reserved: 1.48046875

************************

Processing - flickr30k

************************



/tmp/ipykernel_17484/1019136155.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(


Streaming output truncated to the last 5000 lines.
Validation Time: 10.88 sec
GPU Memory Used: 1.25 GB
✓ Best model saved(Val Loss: 2.9521)

================ Epoch 3/50 ================
Batch 01/94 | Loss: 2.8171 | Time: 0.60s
Batch 02/94 | Loss: 2.9654 | Time: 0.55s
Batch 03/94 | Loss: 2.8666 | Time: 0.57s
Batch 04/94 | Loss: 3.0468 | Time: 0.55s
Batch 05/94 | Loss: 2.9217 | Time: 0.54s
Batch 06/94 | Loss: 2.7525 | Time: 0.54s
Batch 07/94 | Loss: 2.9978 | Time: 0.54s
Batch 08/94 | Loss: 3.0216 | Time: 0.55s
Batch 09/94 | Loss: 3.1877 | Time: 0.54s
Batch 10/94 | Loss: 2.9027 | Time: 0.55s
Batch 11/94 | Loss: 2.9529 | Time: 0.54s
Batch 12/94 | Loss: 2.8727 | Time: 0.55s
Batch 13/94 | Loss: 2.8741 | Time: 0.55s
Batch 14/94 | Loss: 2.8070 | Time: 0.54s
Batch 15/94 | Loss: 2.8861 | Time: 0.55s
Batch 16/94 | Loss: 2.9064 | Time: 0.54s
Batch 17/94 | Loss: 2.9278 | Time: 0.55s
Batch 18/94 | Loss: 2.6995 | Time: 0.55s
Batch 19/94 | Loss: 2.8327 | Time: 0.54s
Batch 20/94 | Loss: 2.8402 | Time: 

,Metric,Image→Text,Text→Image
0,Recall@1,0.258998,0.187516
1,Recall@5,0.529323,0.428442
2,Recall@10,0.649132,0.546690
3,MRR,0.385791,0.303594



Memory after cleanup
Allocated: 0.7816190719604492
Reserved: 1.48046875
